# 02 — Swaption Market Instruments

**Goal:** convert ATM swaption volatility quotes into calibration instruments. We use `SwaptionHelper` so that each quote can be compared with the value produced by a candidate short-rate model.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import QuantLib as ql

from src.rates_project import *
set_evaluation_date()
print("QuantLib version:", ql.__version__)
print("Evaluation date:", ql.Settings.instance().evaluationDate)

QuantLib version: 1.43
Evaluation date: January 15th, 2026


In [2]:
curve, curve_handle, _ = build_curve()
helpers, swaption_meta = build_swaption_helpers(curve_handle)
print("Number of calibration instruments:", len(helpers))

Number of calibration instruments: 12


## 1. Volatility matrix

In [3]:
vol_df = swaption_vol_table()
vol_df.pivot(index="expiry", columns="tenor", values="vol")

tenor,10Y,1Y,2Y,5Y
expiry,,,,
1Y,0.218,0.235,0.230,0.223
2Y,0.215,0.232,0.227,0.220
5Y,0.208,0.224,0.219,0.213


## 2. Calibration targets

The helper objects translate the quoted implied volatility into a market calibration target under their pricing conventions. This separates **market quoting** from **model calibration**.

In [4]:
market_targets = swaption_market_table(swaption_meta)
market_targets

,swaption,expiry,tenor,market_vol,market_value
0,1Yx1Y,1Y,1Y,0.235,0.002113
1,1Yx2Y,1Y,2Y,0.230,0.004181
2,1Yx5Y,1Y,5Y,0.223,0.010311
3,1Yx10Y,1Y,10Y,0.218,0.019762
4,2Yx1Y,2Y,1Y,0.232,0.002999
5,2Yx2Y,2Y,2Y,0.227,0.005955
6,2Yx5Y,2Y,5Y,0.220,0.014364
7,2Yx10Y,2Y,10Y,0.215,0.027202
8,5Yx1Y,5Y,1Y,0.224,0.004632
9,5Yx2Y,5Y,2Y,0.219,0.008979
